Splitting the dataset to train test and valifdation for MSVD 


In [ ]:
import os
import random
import shutil
import glob
import pandas as pd
from collections import defaultdict
import csv

# ====== CONFIGURATION ======
video_dir = fr"D:\major project\Video_captioning_code\MSVD\videos\YouTubeClips"
annotations_path = fr"D:\major project\Video_captioning_code\MSVD\nepali_captions_only.csv"
output_dir = fr"D:\major project\Video_captioning_code\msvd_splitted"

train_count = 1200
val_count = 670
test_count = 100

# ====== STEP 1: LOAD ANNOTATIONS FROM CSV ======
annotations_df = pd.read_csv(annotations_path)
print("Sample from annotations:")
print(annotations_df.head())

annotations_dict = defaultdict(list)
for _, row in annotations_df.iterrows():
    video_id = str(row['video_id']).strip()
    caption = str(row['nepali_captions']).strip()
    if video_id and caption:
        annotations_dict[video_id].append(caption)

print(f"Loaded {len(annotations_dict)} unique video IDs from annotation CSV.")

# ====== STEP 2: VALIDATE AND SPLIT IDS ======
all_video_ids = list(annotations_dict.keys())
total_available = len(all_video_ids)

assert total_available >= (train_count + val_count + test_count), \
    f"Only {total_available} videos available, but {train_count + val_count + test_count} needed."

random.seed(42)
random.shuffle(all_video_ids)

train_ids = all_video_ids[:train_count]
val_ids = all_video_ids[train_count:train_count + val_count]
test_ids = all_video_ids[train_count + val_count:train_count + val_count + test_count]

splits = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids
}

# ====== STEP 3: COPY VIDEOS & WRITE CAPTIONS TO CSV ======
for split_name, split_ids in splits.items():
    print(f"\nProcessing {split_name} split with {len(split_ids)} videos.")
    split_video_dir = os.path.join(output_dir, split_name, 'videos')
    os.makedirs(split_video_dir, exist_ok=True)

    copied_entries = []

    for idx, video_id in enumerate(split_ids, 1):
        pattern = os.path.join(video_dir, f"{video_id}.*")  # force extension match
        matched_files = glob.glob(pattern)

        print(f"Looking for: {pattern}")
        print(f"Matches: {matched_files}")

        if matched_files:
            for src_path in matched_files:
                filename = os.path.basename(src_path)
                new_filename = f"{idx:05d}_{filename}"
                dst_path = os.path.join(split_video_dir, new_filename)
                shutil.copy2(src_path, dst_path)
                copied_entries.append((f"{idx:05d}", video_id))
        else:
            print(f"Warning: No video file found starting with ID: {video_id}")

    print(f"Copied {len(copied_entries)} videos for {split_name} split.")

    # Write captions to CSV
    caption_output_path = os.path.join(output_dir, split_name, f"{split_name}.csv")
    rows = []
    for index_prefix, video_id in copied_entries:
        captions = annotations_dict.get(video_id, [])
        for caption in captions:
            rows.append([f"{index_prefix}_{video_id}", caption])

    print(f"Writing {len(rows)} captions to {caption_output_path}")
    with open(caption_output_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['video_id', 'nepali_caption'])  # header
        writer.writerows(rows)

print("\nAll splits processed and saved as CSV files successfully.")


📄 Sample from annotations:
            video_id                     nepali_captions
0  -4wsuPCjDBc_5_15            एउटा चिपमङ्कले खाइरहेको छ
1  -4wsuPCjDBc_5_15       एउटा चिपमङ्कले बदाम खाइरहेको छ
2  -4wsuPCjDBc_5_15         एउटा चिपमङ्कले नट खाइरहेको छ
3  -4wsuPCjDBc_5_15          एउटा गिलहरीले नट खाइरहेको छ
4  -4wsuPCjDBc_5_15   एउटा गिलहरीले पुरै बदाम खाइरहेको छ
✅ Loaded 1970 unique video IDs from annotation CSV.

📝 Processing train split with 1200 videos.
🔍 Looking for: D:\major project\Video_captioning_code\MSVD\videos\YouTubeClips\nULE40HEWpA_5_11.*
🔎 Matches: ['D:\\major project\\Video_captioning_code\\MSVD\\videos\\YouTubeClips\\nULE40HEWpA_5_11.avi']
🔍 Looking for: D:\major project\Video_captioning_code\MSVD\videos\YouTubeClips\0lh_UWF9ZP4_199_207.*
🔎 Matches: ['D:\\major project\\Video_captioning_code\\MSVD\\videos\\YouTubeClips\\0lh_UWF9ZP4_199_207.avi']
🔍 Looking for: D:\major project\Video_captioning_code\MSVD\videos\YouTubeClips\BGG0uYWZBdw_6_12.*
🔎 Matches: ['D:\\major 

Verify if there is unique ids equal in captions and videos or not in each folders

In [12]:
import os
import pandas as pd

splits = ["train", "val", "test"]
base_dir = r"D:\major project\Video_captioning_code\msvd_splitted"

for split in splits:
    print(f"\nChecking split: {split}")
    video_dir = os.path.join(base_dir, split, "videos")
    csv_path = os.path.join(base_dir, split, f"{split}.csv")

    # List all video files (remove extension for matching)
    video_files = [os.path.splitext(f)[0] for f in os.listdir(video_dir)]
    print(f"Videos found: {len(video_files)}")

    # Load captions
    df = pd.read_csv(csv_path)
    caption_ids = set(df['video_id'].unique())
    print(f"Captions found for: {len(caption_ids)} videos")

    # Check if all videos in folder have captions
    missing_caption = [vid for vid in video_files if vid not in caption_ids]
    if missing_caption:
        print(f"{len(missing_caption)} videos have no caption entry in CSV:")
        print(missing_caption[:5])
    else:
        print("All videos in folder have captions.")

    # Check if all captioned video_ids have actual videos
    missing_video = [vid for vid in caption_ids if vid not in video_files]
    if missing_video:
        print(f"{len(missing_video)} caption entries refer to missing videos:")
        print(missing_video[:5])
    else:
        print("All caption entries have matching videos.")



Checking split: train
Videos found: 1200
Captions found for: 1200 videos
All videos in folder have captions.
All caption entries have matching videos.

Checking split: val
Videos found: 670
Captions found for: 670 videos
All videos in folder have captions.
All caption entries have matching videos.

Checking split: test
Videos found: 100
Captions found for: 100 videos
All videos in folder have captions.
All caption entries have matching videos.


Translation using NLLB

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path
from tqdm.auto import tqdm                    # NEW ✨

# ----------------------- 1. CONFIG -----------------------
model_name   = "facebook/nllb-200-distilled-600M"
src_lang     = "eng_Latn"
tgt_lang     = "npi_Deva"
input_file   = Path("captions.txt")     # <- local file, adjust path
output_file  = Path("captions_ne.txt")
batch_size   = 32

# ----------------------- 2. SETUP ------------------------
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

tokenizer.src_lang = src_lang
forced_bos_id      = tokenizer.convert_tokens_to_ids(tgt_lang)

# ----------------------- 3. LOAD DATA --------------------
with input_file.open(encoding="utf-8") as f:
    captions = [line.strip() for line in f if line.strip()]

total_lines = len(captions)

# ----------------------- 4. TRANSLATE --------------------
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

nepali_captions = []
model.eval()

with torch.no_grad():
    for batch in tqdm(chunks(captions, batch_size),
                      total=(total_lines + batch_size - 1) // batch_size,
                      desc="Translating",
                      unit="batch"):
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_length=30
        )
        nepali_captions.extend(
            tokenizer.batch_decode(generated, skip_special_tokens=True)
        )

# ----------------------- 5. SAVE OUTPUT ------------------
with output_file.open("w", encoding="utf-8") as f:
    for line in nepali_captions:
        f.write(line + "\n")

print(f"✅  Translated {total_lines} captions → {output_file.resolve()}")
